In [1]:
# Economics Project – Extract BACI bilateral total trade
# Input : energy_project BACI_HS92 files (same paths as your oil notebook)
# Output: economics_project/data/raw/baci/

from pathlib import Path
import pandas as pd
import numpy as np

# ---------- Paths ----------
BACI_DIR = Path(r"D:\univercity\PROJECTS_U\energy_project\data\raw\BACI_HS92_V202601")
COUNTRY_FILE = Path(r"D:\univercity\PROJECTS_U\energy_project\data\raw\country_codes_V202601.csv")
if not COUNTRY_FILE.exists():
    COUNTRY_FILE = BACI_DIR / "country_codes_V202601.csv"

OUT_DIR = Path(r"D:\univercity\omid_project\economics_project\data\raw\baci")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 1995, 2024   # later can restrict to 2000-2020

print("=" * 60)
print("BACI → Economics | Total Bilateral Trade Extraction")
print("=" * 60)
print(f"BACI dir : {BACI_DIR}")
print(f"Output   : {OUT_DIR}")

# ---------- Country mapping ----------
countries = pd.read_csv(COUNTRY_FILE, dtype=str)
countries["country_code"] = pd.to_numeric(countries["country_code"], errors="coerce")
countries = countries.dropna(subset=["country_code"])
countries["country_code"] = countries["country_code"].astype(int)

code_to_iso2 = dict(zip(countries["country_code"], countries["country_iso2"]))
code_to_iso3 = dict(zip(countries["country_code"], countries["country_iso3"]))
code_to_name = dict(zip(countries["country_code"], countries["country_name"]))

# ---------- Read all years & aggregate to country-pair ----------
files = sorted(BACI_DIR.glob("BACI_HS92_Y*_V202601.csv"))
print(f"Found {len(files)} annual files")

chunks = []
for f in files:
    year = int(f.name.split("_Y")[1][:4])
    if year < YEAR_MIN or year > YEAR_MAX:
        continue
    print(f"Processing {f.name} ...", end=" ")
    df = pd.read_csv(f, usecols=["t", "i", "j", "v", "q"])
    df["v"] = pd.to_numeric(df["v"], errors="coerce")
    df["q"] = pd.to_numeric(df["q"], errors="coerce")
    df = df.dropna(subset=["v"])
    df = df[df["i"] != df["j"]]
    
    # aggregate all products → one edge per exporter-importer-year
    g = df.groupby(["t", "i", "j"], as_index=False).agg(
        value_kusd=("v", "sum"),
        qty_tonnes=("q", "sum")
    )
    print(f"{len(g):,} country-pair edges")
    chunks.append(g)

if not chunks:
    raise ValueError("No data extracted.")

out = pd.concat(chunks, ignore_index=True)

# map codes
out["i"] = pd.to_numeric(out["i"], errors="coerce").astype("Int64")
out["j"] = pd.to_numeric(out["j"], errors="coerce").astype("Int64")

out["exporter_iso2"] = out["i"].map(code_to_iso2)
out["exporter_iso3"] = out["i"].map(code_to_iso3)
out["exporter_name"] = out["i"].map(code_to_name)
out["importer_iso2"] = out["j"].map(code_to_iso2)
out["importer_iso3"] = out["j"].map(code_to_iso3)
out["importer_name"] = out["j"].map(code_to_name)

out = out.sort_values(["t", "i", "j"]).reset_index(drop=True)

# save
csv_path = OUT_DIR / "baci_total_trade_bilateral.csv"
pq_path  = OUT_DIR / "baci_total_trade_bilateral.parquet"

out.to_csv(csv_path, index=False)
out.to_parquet(pq_path, index=False)

print("\nDone.")
print(f"Total rows     : {len(out):,}")
print(f"Years          : {out['t'].min()} → {out['t'].max()}")
print(f"Exporters      : {out['i'].nunique()}")
print(f"Importers      : {out['j'].nunique()}")
print(f"Mapped exp ISO2: {out['exporter_iso2'].notna().sum():,}")
print(f"Mapped imp ISO2: {out['importer_iso2'].notna().sum():,}")
print(f"Saved:\n  {csv_path}\n  {pq_path}")
print(out.head(6))

BACI → Economics | Total Bilateral Trade Extraction
BACI dir : D:\univercity\PROJECTS_U\energy_project\data\raw\BACI_HS92_V202601
Output   : D:\univercity\omid_project\economics_project\data\raw\baci
Found 30 annual files
Processing BACI_HS92_Y1995_V202601.csv ... 19,015 country-pair edges
Processing BACI_HS92_Y1996_V202601.csv ... 20,499 country-pair edges
Processing BACI_HS92_Y1997_V202601.csv ... 21,558 country-pair edges
Processing BACI_HS92_Y1998_V202601.csv ... 22,459 country-pair edges
Processing BACI_HS92_Y1999_V202601.csv ... 23,059 country-pair edges
Processing BACI_HS92_Y2000_V202601.csv ... 26,049 country-pair edges
Processing BACI_HS92_Y2001_V202601.csv ... 26,783 country-pair edges
Processing BACI_HS92_Y2002_V202601.csv ... 27,364 country-pair edges
Processing BACI_HS92_Y2003_V202601.csv ... 27,732 country-pair edges
Processing BACI_HS92_Y2004_V202601.csv ... 28,574 country-pair edges
Processing BACI_HS92_Y2005_V202601.csv ... 28,847 country-pair edges
Processing BACI_HS9

In [2]:
# Economics Project – Clean BACI total trade + build edges & node attrs

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
BACI_RAW = PROJECT / "data" / "raw" / "baci" / "baci_total_trade_bilateral.parquet"
OUT_DIR  = PROJECT / "data" / "raw" / "baci"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 2000, 2020   # overlap with JST core period + modern years

print("=" * 60)
print("BACI Clean + Trade Edges / Node Attributes")
print("=" * 60)

# ---------- Load ----------
df = pd.read_parquet(BACI_RAW)
print(f"Raw bilateral rows: {len(df):,}")
print(f"Years: {df['t'].min()} → {df['t'].max()}")

# ---------- Clean ----------
edges = df[(df["t"] >= YEAR_MIN) & (df["t"] <= YEAR_MAX)].copy()
edges = edges[edges["i"] != edges["j"]]
edges = edges.dropna(subset=["value_kusd"])
edges = edges[edges["value_kusd"] > 0]

# drop unmapped ISO2
edges = edges.dropna(subset=["exporter_iso2", "importer_iso2"])

print(f"After clean/filter: {len(edges):,}")
print(f"Years kept: {edges['t'].min()} → {edges['t'].max()}")
print(f"Unique exporters ISO2: {edges['exporter_iso2'].nunique()}")
print(f"Unique importers ISO2: {edges['importer_iso2'].nunique()}")

# ---------- Trade edges table ----------
trade_edges = pd.DataFrame({
    "year": edges["t"].astype(int),
    "source_code": edges["i"].astype(int),
    "target_code": edges["j"].astype(int),
    "source_iso2": edges["exporter_iso2"],
    "target_iso2": edges["importer_iso2"],
    "source_iso3": edges["exporter_iso3"],
    "target_iso3": edges["importer_iso3"],
    "source_name": edges["exporter_name"],
    "target_name": edges["importer_name"],
    "weight_value_kusd": edges["value_kusd"].astype(float),
    "weight_tonnes": edges["qty_tonnes"].astype(float),
    "category": "bilateral_trade",
    "flow": "TOTAL_EXPORT_TO",
    "edge_type": "total_trade"
})
trade_edges = trade_edges.sort_values(["year", "source_code", "target_code"]).reset_index(drop=True)

# ---------- Node attributes (annual export/import totals) ----------
exp = trade_edges.groupby(["year", "source_iso2"], as_index=False)["weight_value_kusd"].sum()
exp = exp.rename(columns={"source_iso2": "country", "weight_value_kusd": "export_kusd"})

imp = trade_edges.groupby(["year", "target_iso2"], as_index=False)["weight_value_kusd"].sum()
imp = imp.rename(columns={"target_iso2": "country", "weight_value_kusd": "import_kusd"})

node_attrs = exp.merge(imp, on=["year", "country"], how="outer").fillna(0.0)
node_attrs["net_trade_kusd"] = node_attrs["export_kusd"] - node_attrs["import_kusd"]
node_attrs["trade_openness_kusd"] = node_attrs["export_kusd"] + node_attrs["import_kusd"]
node_attrs["is_exporter"] = (node_attrs["export_kusd"] > 0).astype(int)
node_attrs["is_importer"] = (node_attrs["import_kusd"] > 0).astype(int)
node_attrs = node_attrs.sort_values(["year", "country"]).reset_index(drop=True)

print(f"\nTrade edges : {trade_edges.shape}")
print(f"Node attrs  : {node_attrs.shape} | countries: {node_attrs['country'].nunique()}")

# ---------- Save ----------
te_csv = OUT_DIR / "trade_edges_total_2000_2020.csv"
te_pq  = OUT_DIR / "trade_edges_total_2000_2020.parquet"
na_csv = OUT_DIR / "node_attributes_trade_2000_2020.csv"
na_pq  = OUT_DIR / "node_attributes_trade_2000_2020.parquet"

trade_edges.to_csv(te_csv, index=False)
trade_edges.to_parquet(te_pq, index=False)
node_attrs.to_csv(na_csv, index=False)
node_attrs.to_parquet(na_pq, index=False)

print("\nSaved:")
print(f"  {te_csv}")
print(f"  {te_pq}")
print(f"  {na_csv}")
print(f"  {na_pq}")

print("\nTrade edges sample:")
print(trade_edges.head(5).to_string(index=False))
print("\nNode attrs sample:")
print(node_attrs.head(5).to_string(index=False))

print("\nCell 2 completed successfully.")

BACI Clean + Trade Edges / Node Attributes
Raw bilateral rows: 870,755
Years: 1995 → 2024
After clean/filter: 621,576
Years kept: 2000 → 2020
Unique exporters ISO2: 228
Unique importers ISO2: 227

Trade edges : (621576, 14)
Node attrs  : (4668, 8) | countries: 228

Saved:
  D:\univercity\omid_project\economics_project\data\raw\baci\trade_edges_total_2000_2020.csv
  D:\univercity\omid_project\economics_project\data\raw\baci\trade_edges_total_2000_2020.parquet
  D:\univercity\omid_project\economics_project\data\raw\baci\node_attributes_trade_2000_2020.csv
  D:\univercity\omid_project\economics_project\data\raw\baci\node_attributes_trade_2000_2020.parquet

Trade edges sample:
 year  source_code  target_code source_iso2 target_iso2 source_iso3 target_iso3 source_name         target_name  weight_value_kusd  weight_tonnes        category            flow   edge_type
 2000            4           12          AF          DZ         AFG         DZA Afghanistan             Algeria             24.8